# 02 — Временной split и baseline

## tl;dr
Zero, fit median и cur_dev_s оценены тем же evaluator на 750 tune и 473 audit строках. На audit MAE: zero 77.33 с, fit median 75.10 с, cur_dev_s 74.19 с. Разбиение фиксировано в artifacts/t3-full-received-v2/evaluation_model/split_manifest.csv.


## Context & Methods
Frozen split по T: fit до 14:00, tune 14:00–17:30, calibration 17:30–20:30, audit после 20:30 6 января 2026. Перед следующей границей предыдущие labels должны созреть до 15-минутного lookback. labels_test ранее виден и не участвует в split.


In [1]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
run_dir = root / os.environ.get('TRANSPORT_MODEL_RUN', 'artifacts/t3-full-received-v2')
sys.path.insert(0, str(root))
from transport_ml.data import read_points
from transport_ml.validation import chronological_blocks, regression_metrics
points = read_points(root / 'data/labels/labels_train.csv', require_target=True)
masks, metadata = chronological_blocks(points)
manifest = pd.read_csv(run_dir / 'evaluation_model/split_manifest.csv', dtype={'sample_id': str})
expected = pd.Series('purged', index=points.index)
for block, rows in masks.items(): expected.iloc[rows] = block
assert manifest.sample_id.tolist() == points.sample_id.tolist() and manifest.split.tolist() == expected.tolist()
display(pd.DataFrame(metadata).T)


,n,purged,T_min,T_max,boundary_start,boundary_end
fit,2540,103,2026-01-06 02:10:00,2026-01-06 13:30:00,None,2026-01-06 14:00:00
tune,750,102,2026-01-06 14:00:00,2026-01-06 17:00:00,2026-01-06 14:00:00,2026-01-06 17:30:00
calibration,392,74,2026-01-06 17:30:00,2026-01-06 20:00:00,2026-01-06 17:30:00,2026-01-06 20:30:00
audit,473,0,2026-01-06 20:30:00,2026-01-07 00:25:00,2026-01-06 20:30:00,None


## Data & Results
MAE ниже — лучше. Все прогнозы используют одну целевую задержку в секундах и одну функцию regression_metrics.


In [2]:
y = points.target_delay_s.to_numpy()
cur = points.cur_dev_s.to_numpy()
fit_median = float(np.median(y[masks['fit']]))
rows = []
for block in ('tune', 'audit'):
    ix = masks[block]
    for name, prediction in {'zero': np.zeros(len(y)), 'fit_median': np.full(len(y), fit_median), 'cur_dev': cur}.items():
        score = regression_metrics(y[ix], prediction[ix], cur[ix], points.iloc[ix].tr_id)
        rows.append({'block': block, 'candidate': name, 'n': score['n'], 'mae_s': score['mae_s'], 'rmse_s': score['rmse_s']})
baseline_table = pd.DataFrame(rows)
display(baseline_table)


,block,candidate,n,mae_s,rmse_s
0,tune,zero,750,82.364000,120.396794
1,tune,fit_median,750,86.572000,118.290834
2,tune,cur_dev,750,77.222667,110.215165
3,audit,zero,473,77.329810,126.641835
4,audit,fit_median,473,75.095137,116.309004
5,audit,cur_dev,473,74.192389,127.782097


## Takeaways
Audit — фиксированный будущий интервал внутри train, но 382/473 его строк ранее встречались в других audit artifacts. Это повторная локальная диагностика, не unseen и не platform readback.
